# 05 — Train RF-DETR

Trains RF-DETR-Medium on the BDD100K clear-weather training split.
RF-DETR uses a **DINOv2 ViT-B/14 backbone** and the `rfdetr` Roboflow package.
This notebook uses the **COCO-format** dataset (see `01_setup_and_data.ipynb`).

**Note:** RF-DETR does NOT use the Ultralytics API — it has its own training interface.

**Prerequisites:** run `01_setup_and_data.ipynb` first.

In [ ]:
!pip install -q rfdetr

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

import torch
from pathlib import Path
from src.train_utils import setup_logging

logger = setup_logging()
COCO_DATA_DIR = Path(os.environ['COCO_DATA_DIR'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('COCO data dir exists:', COCO_DATA_DIR.exists())

In [ ]:
from rfdetr import RFDETRMedium

model = RFDETRMedium()

In [ ]:
# TODO: Verify the rfdetr API for the installed version.
#       The exact argument names may differ — check: help(model.train)
#
# RF-DETR expects COCO-format data:
#   COCO_DATA_DIR/train_clear/   — images + annotations.json
#   COCO_DATA_DIR/val_clear/     — images + annotations.json

CHECKPOINT_DIR = Path('../checkpoints/rfdetr')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

model.train(
    dataset=str(COCO_DATA_DIR / 'train_clear'),
    val_dataset=str(COCO_DATA_DIR / 'val_clear'),
    epochs=100,
    batch_size=16,
    # TODO: set output_dir if the rfdetr API supports it
)
print('Training complete.')

In [ ]:
# TODO: Save final checkpoint once training is done.
# RF-DETR's save API may differ — check documentation.
#
# Example (adjust to actual API):
#   model.save(str(CHECKPOINT_DIR / 'best.pt'))

import json
summary = {'model': 'rfdetr', 'epochs': 100}
(CHECKPOINT_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2))
print('Summary saved.')